# 2026.10.19(월) 22h30~

In [2]:
%pip install -U openai langgraph requests python-dotenv

  Using cached openai-3.35.1-py3-none-any.whl.metadata (44 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 13.3 MB/s  0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 3.31.0
    Uninstalling openai-3.31.0:
      Successfully uninstalled openai-3.31.0
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os                                           # 환경 변수 읽기
import json                                         # 검색 결과를 JSON 문자열로 묶어 전달
from pathlib import Path                            # 파일 경로 다루기
from datetime import datetime, timezone, timedelta  # 검색 시각 기록 (한국 시간)
from typing import TypedDict                        # 그래프 상태의 형태 정의
from urllib.parse import urlsplit                   # 검색 결과 링크가 올바른 주소인지 확인

import requests                     # SerpAPI에 HTTP 요청
from dotenv import load_dotenv      # .env 파일 읽기
# OpenAI 클라이언트와 오류 종류별 클래스
from openai import (
    OpenAI, OpenAIError, AuthenticationError, PermissionDeniedError,
    RateLimitError, APITimeoutError, APIConnectionError,
    BadRequestError, NotFoundError,
)
# StateGraph: 그래프 생성 / START, END: 그래프의 시작점과 끝점
from langgraph.graph import StateGraph, START, END

BASE = Path.cwd()
# 현재 폴더의 .env를 먼저 찾고, 없으면 상위 폴더의 .env 사용
load_dotenv(BASE / ".env", override=False) or load_dotenv(BASE.parent / ".env", override=False)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
# SerpAPI 키는 SERPAPI_KEY를 먼저 찾고, 없으면 SERPAPI_API_KEY 사용
SERPAPI_KEY = (
    os.getenv("SERPAPI_KEY", "").strip()
    or os.getenv("SERPAPI_API_KEY", "").strip()
)
TEXT_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini").strip()

# 키가 없거나 .env.example의 예시 문구를 그대로 둔 경우 실행을 멈춤
if not OPENAI_API_KEY or OPENAI_API_KEY.startswith("운영팀에서_"):
    raise ValueError(".env의 OPENAI_API_KEY에 실제 키를 설정하세요.")
if not SERPAPI_KEY or SERPAPI_KEY.startswith("운영팀에서_"):
    raise ValueError(".env의 SERPAPI_KEY에 실제 키를 설정하세요.")
if not TEXT_MODEL:
    raise ValueError("OPENAI_MODEL이 비어 있습니다.")

# API 클라이언트 생성 (timeout: 60초까지 응답 대기, max_retries: 실패 시 2번까지 재시도)
client = OpenAI(api_key=OPENAI_API_KEY, timeout=60.0, max_retries=2)
print("작업 폴더:", BASE)
print("대화 모델:", TEXT_MODEL)
print("설정 준비 완료 — 인증·권한은 실제 요청 시 확인됩니다.")

작업 폴더: /Users/greenpianorabbit/Documents/Development/ict_innovation_sq_voice_ai/016_rag
대화 모델: gpt-4o-mini
설정 준비 완료 — 인증·권한은 실제 요청 시 확인됩니다.


In [4]:
# 검색 결과 1건의 형태 (제목, 요약, 링크, 작성일)
class SearchResult(TypedDict):
    title: str
    snippet: str
    link: str
    date: str


# 그래프의 상태: 노드들이 함께 읽고 쓰는 데이터
# total=False: 모든 항목을 한꺼번에 채우지 않아도 됨 (각 노드는 바꾼 항목만 반환)
class QAState(TypedDict, total=False):
    question: str                          # 사용자 질문
    search_results: list[SearchResult]     # 검색 결과 목록
    searched_at: str                       # 검색한 시각
    answer: str                            # 최종 답변
    error: str                             # 오류 메시지 (없으면 빈 문자열)


# 검색 단계에서 발생한 오류를 다른 오류와 구분하기 위한 클래스
class SearchError(RuntimeError):
    """사용자에게 표시할 수 있는 검색 오류."""


def clean_text(value):
    # 문자열이면 앞뒤 공백을 제거하고, 문자열이 아니면(None 등) 빈 문자열로 바꿈
    return value.strip() if isinstance(value, str) else ""